# DDT Analyzer
### Which r/ssbm users have the biggest effect on Daily Discussion Thread activity?

This notebook scrapes DDT posts and comments from r/ssbm, then uses statistical analysis to rank each user's effect on comment count — controlling for day of week, time trends, seasonal patterns, and community events.

**Just click Runtime > Run all (or Ctrl+F9) to run everything.**

---
## Step 1: Setup
Clone the repo and install dependencies.

In [ ]:
import os

if not os.path.exists('DDTAnalyzer'):
    !git clone -b claude/analyze-melee-user-presence-VeBg6 https://github.com/nwlb91/DDTAnalyzer.git
else:
    print('Already cloned')

%cd DDTAnalyzer
!pip install -q -r requirements.txt

---
## Step 2: Settings
Adjust these before running if you want different thresholds.

In [ ]:
#@title Analysis Settings { display-mode: "form" }

#@markdown **User filters** — who qualifies for the tier list:
MIN_DDTS = 5        #@param {type: "integer"}
MIN_COMMENTS = 10   #@param {type: "integer"}

#@markdown **Display:**
TOP_N = 50           #@param {type: "integer"}

#@markdown **Ridge regression:**
RIDGE_ALPHA = 10.0   #@param {type: "number"}

#@markdown **Scraping:**
MAX_SEARCH_PAGES = 20  #@param {type: "integer"}

print(f'Settings: min_ddts={MIN_DDTS}, min_comments={MIN_COMMENTS}, top_n={TOP_N}, ridge_alpha={RIDGE_ALPHA}, max_pages={MAX_SEARCH_PAGES}')

---
## Step 3: Scrape DDTs from Reddit
This fetches DDT posts and their comments from r/ssbm. Takes ~30-45 minutes for a full scrape due to Reddit rate limits.

Data is cached to disk, so re-running this cell will skip already-fetched posts.

In [ ]:
from scrape import scrape_all

posts, comments = scrape_all(max_search_pages=MAX_SEARCH_PAGES)

total = sum(len(c) for c in comments.values())
print(f'\nDone! {len(posts)} DDTs, {total:,} comments scraped.')

---
## Step 4: Run Analysis & Generate Tier List
This controls for confounders and ranks every user.

In [ ]:
from analyze import run_analysis

tiered_df = run_analysis(
    min_ddts=MIN_DDTS,
    min_comments=MIN_COMMENTS,
    ridge_alpha=RIDGE_ALPHA,
    top_n=TOP_N,
)

---
## Step 5: Explore Results
The full results are in a DataFrame you can filter, sort, and search.

In [ ]:
# Show all S and A tier users
top_tiers = tiered_df[tiered_df['tier'].isin(['S', 'A'])]
print(f'S and A tier users: {len(top_tiers)}')
top_tiers[['user', 'tier', 'effect_residual', 'ridge_coef', 'n_ddts_present', 'presence_rate', 'avg_own_comments', 'p_value_corrected']]

In [ ]:
# Search for a specific user (change the name below)
search_user = 'example_username'  #@param {type: "string"}

match = tiered_df[tiered_df['user'].str.contains(search_user, case=False, na=False)]
if match.empty:
    print(f'No user matching "{search_user}" found in the tier list.')
    print(f'(They may not meet the minimum thresholds: {MIN_DDTS} DDTs, {MIN_COMMENTS} comments)')
else:
    print(match[['user', 'tier', 'effect_residual', 'ridge_coef', 'n_ddts_present', 'presence_rate', 'avg_own_comments']].to_string(index=False))

In [ ]:
# Download the full tier list as CSV
try:
    from google.colab import files
    tiered_df.to_csv('ddt_tier_list.csv', index=False)
    files.download('ddt_tier_list.csv')
    print('Downloading ddt_tier_list.csv...')
except ImportError:
    tiered_df.to_csv('ddt_tier_list.csv', index=False)
    print('Saved to ddt_tier_list.csv')